In [5]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog
import os
from pathlib import Path
from datetime import datetime
import numpy as np

def select_csv_files():
    """Opens a dialog to select multiple CSV result files."""
    root = tk.Tk()
    root.withdraw()
    
    default_dir = Path(os.getcwd()) / 'Results'
    if not default_dir.exists(): default_dir = os.getcwd()
        
    print(f"Please select the result CSV files to compare (starting in {default_dir})...")
    file_paths = filedialog.askopenfilenames(
        title="Select Result CSVs",
        filetypes=[("CSV Files", "*.csv")],
        initialdir=default_dir
    )
    return file_paths

def parse_metadata(csv_path):
    metadata = {}
    try:
        with open(csv_path, 'r') as f:
            for line in f:
                if line.startswith('#'):
                    parts = line.strip().split(':', 1)
                    if len(parts) == 2:
                        key = parts[0].replace('#', '').strip()
                        value = parts[1].strip()
                        metadata[key] = value
                else:
                    break 
    except Exception:
        pass
    return metadata

# --- 1. Select Files ---
csv_paths = select_csv_files()

if not csv_paths:
    print("No files selected.")
else:
    print(f"Selected {len(csv_paths)} files for comparison.")
    
    merged_df = None
    model_cols = [] # To keep track of which columns are model predictions
    model_meta = {} # Store metadata for header
    
    # --- 2. Load and Merge ---
    for i, path_str in enumerate(csv_paths):
        path = Path(path_str)
        meta = parse_metadata(path)
        arch = meta.get('Model Architecture', path.stem.replace('_results', ''))
        weights = meta.get('Model Weights', 'unknown')
        
        # Create unique column ID
        col_id = f"{arch}_{i+1}"
        model_cols.append(f"Class_{col_id}") # Track the class column name
        
        # Store for header
        model_meta[f"Class_{col_id}"] = f"{arch} ({weights})"
        
        print(f"Loading results for: {col_id}")
        
        try:
            df = pd.read_csv(path, comment='#')
            df = df.rename(columns={
                "Predicted_Class": f"Class_{col_id}",
                "Confidence": f"Conf_{col_id}"
            })
            
            if merged_df is None:
                merged_df = df
            else:
                merged_df = pd.merge(merged_df, df, on="Filename", how="outer")
                
        except Exception as e:
            print(f"Error reading {path.name}: {e}")

    if merged_df is not None:
        # --- 3. Analyze Consensus ---
        print("\nAnalyzing consensus...")
        
        # Calculate Status
        def check_agreement(row):
            predictions = set(row[col] for col in model_cols if pd.notna(row[col]))
            if len(predictions) == 0: return "No Predictions"
            elif len(predictions) == 1: return "Agreed"
            else: return "DISAGREEMENT"

        merged_df["Status"] = merged_df.apply(check_agreement, axis=1)
        
        # Calculate Likely Class (Majority Vote)
        merged_df["Likely_Class"] = merged_df[model_cols].mode(axis=1)[0]
        
        # Reorder columns
        cols = ["Filename", "Status", "Likely_Class"] + [c for c in merged_df.columns if c not in ["Filename", "Status", "Likely_Class"]]
        merged_df = merged_df[cols]

        # --- 4. Calculate Statistics (vs Majority Vote) ---
        stats_rows = []
        
        # A. Overall Accuracy
        row_total = ["Total Accuracy (vs Consensus)"]
        total_files = len(merged_df)
        
        for col in model_cols:
            matches = (merged_df[col] == merged_df["Likely_Class"]).sum()
            acc = matches / total_files
            row_total.append(f"{matches}/{total_files} ({acc:.1%})")
            
        stats_rows.append(row_total)
        
        # B. Per-Class Accuracy
        all_classes = sorted(merged_df["Likely_Class"].dropna().unique())
        for cls in all_classes:
            row_cls = [f"Class: {cls}"]
            cls_mask = merged_df["Likely_Class"] == cls
            cls_total = cls_mask.sum()
            
            for col in model_cols:
                # Count where model matched the class AND consensus was that class
                matches = ((merged_df[col] == cls) & cls_mask).sum()
                acc = matches / cls_total if cls_total > 0 else 0
                row_cls.append(f"{matches}/{cls_total} ({acc:.1%})")
            
            stats_rows.append(row_cls)

        # --- 5. Style the DataFrame ---
        def highlight_cells(row):
            styles = [''] * len(row)
            
            # Highlight Status
            status_idx = list(row.index).index('Status')
            if row['Status'] == 'DISAGREEMENT':
                styles[status_idx] = 'background-color: #ffffcc; font-weight: bold' # Yellow
            elif row['Status'] == 'Agreed':
                styles[status_idx] = 'background-color: #e6ffcc' # Light Green

            # Highlight Disagreements (Outliers)
            likely = row['Likely_Class']
            for i, col_name in enumerate(row.index):
                if col_name in model_cols:
                    val = row[col_name]
                    if pd.notna(val) and val != likely:
                        styles[i] = 'background-color: #ffcccc; color: red' # Red warning

            return styles

        styled_df = merged_df.style.apply(highlight_cells, axis=1)

        # --- 6. Save to Excel with Custom Header ---
        output_filename = "Model_Comparison_Stats.xlsx"
        save_path = Path(csv_paths[0]).parent / output_filename
        
        print(f"Saving detailed report to: {save_path}")
        
        with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
            workbook = writer.book
            worksheet = workbook.create_sheet("Comparison")
            writer.book.active = worksheet
            
            # --- Write Header Block ---
            row_ptr = 1
            worksheet.cell(row=row_ptr, column=1, value="--- MODEL COMPARISON REPORT ---")
            row_ptr += 1
            worksheet.cell(row=row_ptr, column=1, value=f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            row_ptr += 2
            
            # Write Model Names
            worksheet.cell(row=row_ptr, column=1, value="Model Details:")
            for i, col in enumerate(model_cols):
                # Align these with the data columns (offset by 3: Filename, Status, Likely_Class)
                worksheet.cell(row=row_ptr, column=i+4, value=model_meta[col])
            row_ptr += 2
            
            # Write Statistics Table
            worksheet.cell(row=row_ptr, column=1, value="--- STATISTICS (Assumes Consensus is Correct) ---")
            row_ptr += 1
            
            for stat_row in stats_rows:
                worksheet.cell(row=row_ptr, column=1, value=stat_row[0])
                for i, val in enumerate(stat_row[1:]):
                    worksheet.cell(row=row_ptr, column=i+4, value=val)
                row_ptr += 1
            
            row_ptr += 2 # Gap before data
            
            # --- Write Main Data ---
            styled_df.to_excel(writer, sheet_name="Comparison", startrow=row_ptr, index=False)
            
        # --- 7. Console Summary ---
        disagreements = merged_df[merged_df["Status"] == "DISAGREEMENT"]
        print(f"\nTotal Files: {len(merged_df)}")
        print(f"Total Disagreements: {len(disagreements)}")
        
        if not disagreements.empty:
            print("\nTop Disagreements (Showing Outliers):")
            print(disagreements[["Filename", "Likely_Class"] + model_cols].head().to_string(index=False))

Please select the result CSV files to compare (starting in /Storage/Files/practicalML/gitlab/practicalml/Results)...
Selected 10 files for comparison.
Loading results for: ,pre-trained_mobilenetv2_1
Loading results for: ,pre-trained_resnet-18_2
Loading results for: ,scratch_densenet-121_3
Loading results for: ,scratch_efficientnetv2_4
Loading results for: ,scratch_googlenet_50epochs_5
Loading results for: ,scratch_googlenet_6
Loading results for: ,scratch_mobilenetv2_7
Loading results for: ,scratch_resnet-18_8
Loading results for: ,scratch_shufflenetv2_9
Loading results for: ,scratch_vgg16_bn_10

Analyzing consensus...
Saving detailed report to: /Storage/Files/practicalML/gitlab/practicalml/Results/Model_Comparison_Stats.xlsx

Total Files: 65
Total Disagreements: 48

Top Disagreements (Showing Outliers):
                                   Filename Likely_Class Class_,pre-trained_mobilenetv2_1 Class_,pre-trained_resnet-18_2 Class_,scratch_densenet-121_3 Class_,scratch_efficientnetv2_4 C